# NLP Core Pipeline Engineering
This notebook demonstrates the core components of an NLP preprocessing pipeline, from data acquisition and cleaning to advanced tokenization and data structuring using spaCy and Pandas.

In [12]:
'''
NLP Preprocessing Pipeline
Author: Jeremy Fields
'''

import re
import requests
import pandas as pd
import spacy
from spacy.tokens import Doc
from spacy.tokenizer import Tokenizer
from spacy.language import Language
from bs4 import BeautifulSoup
from pprint import pprint

## 1. Data Acquisition
First, we'll fetch the content of an article from the FAO newsroom related to world food prices. We save this locally as an HTML file for consistent processing and to avoid repeated network requests.

In [13]:
URL = "https://www.fao.org/newsroom/detail/world-food-prices-dip-in-december/en"
page = requests.get(URL)

# Save the binary content to a local file
with open("fao-food-prices.html", "wb") as f:
    f.write(page.content)

print("File saved as fao-food-prices.html")

File saved as fao-food-prices.html


In [14]:
with open("fao-food-prices.html", encoding="utf8") as html_file:
    html_content = html_file.read()
pprint(html_content[:1500])

(' <!DOCTYPE html> <html lang="en" > <head> <meta charset="utf-8" /> <meta '
 'name="viewport" content="width=device-width, initial-scale=1, '
 'shrink-to-fit=no"> <title>\n'
 '\tWorld food prices dip in December\n'
 '</title> <script '
 'src="/ScriptResource.axd?d=6DQe8ARl7A9TiuWej5ttCsl0UxczkFZDbeL5SW9kKwZezKThjMd6CKk80af9FalSKa-iav7TTDncR2lY8pDjm3GfAJE4PtQDLQKTHvlXubFpiBl5L2i8chWLcfOXJghIDWVmV7HDcMgBha1sHF1HkvsG2gh96v6WMgcTICorlXSyCWKJN2ZyaRZDDb4Aqzqr0&amp;t=5dc176bf" '
 'type="text/javascript"></script><script '
 'src="/ScriptResource.axd?d=74FHISOx3fOPKwLxL0RMYh70ezZ-ORxFxHKxtz0UOPg9EUQxTPN8-UKE4lkI6c_x7Y5_KWlWwAWxLFKVT-LYvCs8ogtaZX_8Ldu_Ha6fqNvKUYz-iwsAD98noLncrVt8xXRvxvQvI8C6OeG6ayptPlJ8Jhe1U-cIG8z7urOA6SjD5WDdFtJKQYNppo3NHS220&amp;t=5dc176bf" '
 'type="text/javascript"></script><script '
 'src="https://cse.google.com/cse.js?cx=018170620143701104933%3Aqq82jsfba7w" '
 'type="text/javascript"></script><link '
 'href="/ResourcePackages/Bootstrap5/assets/dist/css/main.min.css?v=5.3.

## 2. Text Extraction and Preprocessing
We define a series of functions to extract the meaningful text from the HTML, clean it, and handle basic formatting like sentence endings and whitespace normalization.

In [15]:
def extract_text(html_content: str) -> str:
    soup = BeautifulSoup(html_content, 'html.parser')
    content = soup.find(id="Contentplaceholder1_C011_Col00")
    # return text content if found, else return empty string
    return content.get_text() if content else ""

In [16]:
text = extract_text(html_content)
text[:580]

'\n\n\n\n\n\n\n\n\n\nWorld food prices dip in December\nFAO Food Price Index ends 2022 lower than a year earlier\n\n\n\n\n                                A farmer in Sicily carrying wheat seeds.\n                             \n\n©FAO/Giorgio Cosulich \n\n\n\n\n06/01/2023\n\n\nRome – The index of world food prices dipped for the ninth consecutive month in December 2022, declining by 1.9 percent from the previous month, the Food and Agriculture Organization of the United Nations (FAO) reported today. The FAO Food Price Index averaged 132.4 points in December, 1.0 percent below its value a year earlier. '

In [17]:
def clean_text(text: str) -> str:
    # Split the text into segments based on newlines
    segments = [s.strip() for s in text.split('\n') if s.strip()]
    # print(segments)
    
    cleaned_segments = []
    for segment in segments:
        # Add a period if it doesn't end with one
        if not segment.endswith(('.', '!', '?')):
            segment += '.'
            
        # Use regex to replace multiple spaces with a single space
        segment = re.sub(r' +', ' ', segment)
        cleaned_segments.append(segment)
        
    # Re-join the cleaned segments into a single string with single space
    cleaned_text = ' '.join(cleaned_segments)
    
    return cleaned_text

In [18]:
cleaned_text = clean_text(text)
cleaned_text[:499]

'World food prices dip in December. FAO Food Price Index ends 2022 lower than a year earlier. A farmer in Sicily carrying wheat seeds. ©FAO/Giorgio Cosulich. 06/01/2023. Rome – The index of world food prices dipped for the ninth consecutive month in December 2022, declining by 1.9 percent from the previous month, the Food and Agriculture Organization of the United Nations (FAO) reported today. The FAO Food Price Index averaged 132.4 points in December, 1.0 percent below its value a year earlier.'

## 3. Spacy Pipeline and NLP Tasks
Now, we leverage `spaCy`'s pre-trained English model to perform typical NLP tasks such as lemmatization, Part-Of-Speech (POS) tagging, and Named Entity Recognition (NER). We then structure this data into a DataFrame for easier analysis and querying.

In [19]:
nlp = spacy.load("en_core_web_sm")

In [20]:
def process_text(text: str, nlp: Language) -> Doc:
    # Process the cleaned text (returning a Doc object)
    return nlp(text)

In [21]:
doc = process_text(cleaned_text, nlp)
all(map(doc.has_annotation, ["LEMMA", "POS", "ENT_TYPE"]))

True

In [22]:
# exploring subset of tokens, lemmas, pos tags, and entities
for token in doc[:20]:
    print(f"Token: {token.text}, Lemma: {token.lemma_}, POS: {token.pos_}, Entity Type: {token.ent_type_}")

Token: World, Lemma: world, POS: NOUN, Entity Type: 
Token: food, Lemma: food, POS: NOUN, Entity Type: 
Token: prices, Lemma: price, POS: NOUN, Entity Type: 
Token: dip, Lemma: dip, POS: VERB, Entity Type: 
Token: in, Lemma: in, POS: ADP, Entity Type: 
Token: December, Lemma: December, POS: PROPN, Entity Type: DATE
Token: ., Lemma: ., POS: PUNCT, Entity Type: 
Token: FAO, Lemma: FAO, POS: PROPN, Entity Type: ORG
Token: Food, Lemma: Food, POS: PROPN, Entity Type: ORG
Token: Price, Lemma: Price, POS: PROPN, Entity Type: 
Token: Index, Lemma: Index, POS: PROPN, Entity Type: 
Token: ends, Lemma: end, POS: VERB, Entity Type: 
Token: 2022, Lemma: 2022, POS: NUM, Entity Type: DATE
Token: lower, Lemma: low, POS: ADJ, Entity Type: 
Token: than, Lemma: than, POS: ADP, Entity Type: 
Token: a, Lemma: a, POS: DET, Entity Type: DATE
Token: year, Lemma: year, POS: NOUN, Entity Type: DATE
Token: earlier, Lemma: early, POS: ADV, Entity Type: DATE
Token: ., Lemma: ., POS: PUNCT, Entity Type: 
Token: A, 

In [23]:
def to_dataframe(doc: Doc) -> pd.DataFrame:
    rows = []
    # get sentence spans from the Doc object
    for sent_id, sent in enumerate(doc.sents):
        # iterate over sentence span object to get token objects
        for token_id, token in enumerate(sent):
            rows.append({
                'sent_id': sent_id,
                'token_id': token_id,
                'text': token.text,
                'lemma': token.lemma_,
                'pos': token.pos_,
                'ent': token.ent_type_
            })

    return pd.DataFrame(rows)

In [24]:
df = to_dataframe(doc)
df[df.sent_id == 1]

,sent_id,token_id,text,lemma,pos,ent
7,1,0,FAO,FAO,PROPN,ORG
8,1,1,Food,Food,PROPN,ORG
9,1,2,Price,Price,PROPN,
10,1,3,Index,Index,PROPN,
11,1,4,ends,end,VERB,
12,1,5,2022,2022,NUM,DATE
13,1,6,lower,low,ADJ,
14,1,7,than,than,ADP,
15,1,8,a,a,DET,DATE
16,1,9,year,year,NOUN,DATE


## 4. Tokenizer Customization
Finally, we can customize the default spaCy tokenizer to handle specific use cases, such as splitting on forward slashes, which might not be handled out-of-the-box. This helps in more accurate tokenization for technical or domain-specific texts.

In [25]:
def customize_tokenizer(nlp: Language) -> Language:
    # add new rule to the tokenizer to split on forward slashes
    infixes = nlp.Defaults.infixes + [r'/']
    infix_re = spacy.util.compile_infix_regex(infixes)
    # replace default infix finditer with the new one that includes the forward slash
    nlp.tokenizer.infix_finditer = infix_re.finditer
    return nlp

In [26]:
customized_nlp = customize_tokenizer(nlp)
doc = process_text(cleaned_text, customized_nlp)
df = to_dataframe(doc)
df[df.sent_id == 4]

,sent_id,token_id,text,lemma,pos,ent
33,4,0,06,06,NUM,DATE
34,4,1,/,/,SYM,DATE
35,4,2,01,01,NUM,DATE
36,4,3,/,/,SYM,DATE
37,4,4,2023,2023,NUM,DATE
38,4,5,.,.,PUNCT,
